# Stationary-phase regulator annotations

Regulator labels do not enter morphology feature selection or UMAP fitting. Overlays reuse the supplied coordinates. The 90% WT KDE boundary is descriptive; pies count each deletion gene once, excluding WT.

In [ ]:
from pathlib import Path
import numpy as np
import numpy.typing
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PACKAGE = Path.cwd()
if not (PACKAGE / "Datasets").is_dir():
    PACKAGE = PACKAGE.parent
DATASETS = PACKAGE / "Datasets"
if not DATASETS.is_dir():
    raise FileNotFoundError("Launch this notebook from the package root or Github folder.")
plt.rcParams.update({"font.family": "Arial", "text.usetex": False})

WT_LABEL = "WT"
WT_CONTOUR_PROBABILITY = 0.90
UMAP_PARAMETERS = {"n_neighbors":10,"min_dist":0.1,"random_state":42}
AXIS_ORDER = ["RpoS", "ppGpp", "RpoS and ppGpp", "No documented association"]
PLOT_METADATA_COLUMNS = ["Metadata_stationary_phase_axis", "Metadata_RpoS_effect", "Metadata_ppGpp_effect"]
import seaborn as sns
book = DATASETS / "Dataset S6.xlsx"
umap_df = pd.read_excel(book, sheet_name="UMAP_plotted")
gene_metadata = pd.read_excel(book, sheet_name="Gene_metadata")
reference_pies = pd.read_excel(book, sheet_name="Pie_plotted")
profile_genes = set(umap_df.loc[umap_df.Metadata_gene.ne(WT_LABEL),"Metadata_gene"])

## Optional normalization and embedding reconstruction
Average well profiles within plate/genotype, normalize against pooled WT profiles, and apply the original feature selection. Default plots retain the supplied coordinates.

In [ ]:
RECOMPUTE_UMAP = False
if RECOMPUTE_UMAP:
    from pycytominer import aggregate, normalize, feature_select
    from umap import UMAP
    rescaled_df = pd.read_excel(DATASETS / "Dataset S3.xlsx", sheet_name="Raw_well_profiles")
    profile_features = [column for column in rescaled_df.columns if not column.startswith('Metadata_') and pd.api.types.is_numeric_dtype(rescaled_df[column])]
    aggregated_df = aggregate(population_df=rescaled_df, strata=['Metadata_plate', 'Metadata_gene'], features=profile_features, operation='mean')
    feature_qc = pd.DataFrame({'Feature': profile_features})
    all_missing = aggregated_df[profile_features].isna().all()
    profile_features = [f for f in profile_features if not all_missing[f]]
    normalized_df = normalize(profiles=aggregated_df, features=profile_features, image_features=False, meta_features=['Metadata_gene', 'Metadata_plate'], samples=f"Metadata_gene == '{WT_LABEL}'", method='mad_robustize', output_file=None)
    normalized_df[profile_features] = normalized_df[profile_features].replace([np.inf, -np.inf], np.nan)
    normalized_features = [column for column in normalized_df.columns if not column.startswith('Metadata_')]
    selected_df = feature_select(profiles=normalized_df, features=normalized_features, image_features=False, samples='all', operation=['drop_outliers', 'variance_threshold', 'correlation_threshold'], na_cutoff=0.5, corr_threshold=0.97, corr_method='pearson', freq_cut=0.01, unique_cut=0.1, outlier_cutoff=100, output_file=None).reset_index(drop=True)
    selected_features = [column for column in selected_df.columns if not column.startswith('Metadata_')]
    if not selected_features:
        raise ValueError('No morphology features retained')
    feature_qc['Selected_for_UMAP'] = feature_qc['Feature'].isin(selected_features)
    model = UMAP(n_components=2, n_neighbors=10, min_dist=0.1, random_state=42, n_jobs=1)
    X = selected_df[selected_features].replace([np.inf,-np.inf],np.nan)
    X = X.fillna(X.median())
    umap_df = selected_df[["Metadata_plate","Metadata_gene"]].copy()
    umap_df[["UMAP1","UMAP2"]] = model.fit_transform(X)
    umap_df = umap_df.merge(gene_metadata,on="Metadata_gene",how="left",validate="many_to_one")

## WT density contour
Use the source 400×400 KDE grid with five covariance-SD padding and a 90% integrated probability threshold.

In [ ]:
from scipy.stats import gaussian_kde
wt = umap_df.loc[umap_df.Metadata_gene.eq(WT_LABEL), ["UMAP1", "UMAP2"]].to_numpy().T
kde = gaussian_kde(wt)
padding = 5 * np.sqrt(np.diag(kde.covariance))
x_grid = np.linspace(wt[0].min()-padding[0], wt[0].max()+padding[0], 400)
y_grid = np.linspace(wt[1].min()-padding[1], wt[1].max()+padding[1], 400)
x_mesh, y_mesh = np.meshgrid(x_grid, y_grid)
values = kde(np.vstack([x_mesh.ravel(),y_mesh.ravel()])).reshape(x_mesh.shape)
pixel_area = (x_grid[1]-x_grid[0])*(y_grid[1]-y_grid[0])
mass = float(values.sum()*pixel_area)
descending = np.sort(values.ravel())[::-1]
cumulative = np.cumsum(descending)*pixel_area/mass
threshold = float(descending[min(np.searchsorted(cumulative,0.9),len(descending)-1)])
wt_density = {"x_mesh":x_mesh,"y_mesh":y_mesh,"values":values,"levels":[threshold],"thresholds":{0.9:threshold},"kde":kde,"integrated_mass":mass}

In [ ]:
METADATA_LABELS = {'Metadata_stationary_phase_axis': 'Stationary-phase axis', 'Metadata_RpoS_effect': 'RpoS effect', 'Metadata_ppGpp_effect': 'ppGpp effect'}
FIXED_PALETTES = {'Metadata_stationary_phase_axis': {'RpoS': '#E69F00', 'ppGpp': '#0072B2', 'RpoS and ppGpp': '#009E73', 'No documented association': '#C7C7C7', 'Unclear': '#7A7A7A', 'Not annotated': '#CC79A7', 'Wild type': '#111111'}, 'Metadata_RpoS_effect': {'activated': '#009E73', 'repressed': '#D55E00', 'mixed or condition-dependent': '#CC79A7', 'no documented evidence': '#C7C7C7', 'Not annotated': '#7A7A7A', 'Wild type': '#111111'}, 'Metadata_ppGpp_effect': {'activated': '#009E73', 'repressed': '#D55E00', 'mixed or condition-dependent': '#CC79A7', 'no documented evidence': '#C7C7C7', 'Not annotated': '#7A7A7A', 'Wild type': '#111111'}}
BACKGROUND_CATEGORIES = ['No documented association', 'no documented evidence', 'none', 'unassigned', 'Unclear', 'Not annotated']

def metadata_display_values(dataframe, column):
    values = dataframe[column].astype('string').str.strip()
    values = values.mask(values.isna() | values.eq(''), 'Not annotated')
    values = values.mask(dataframe['Metadata_gene'].eq(WT_LABEL), 'Wild type')
    return values

def ordered_categories(values):
    observed = set(values.dropna().astype(str))
    background = [category for category in BACKGROUND_CATEGORIES if category in observed]
    foreground = sorted(observed - set(background) - {'Wild type'})
    wild_type = ['Wild type'] if 'Wild type' in observed else []
    return background + foreground + wild_type

def category_palette(column, categories):
    fixed = FIXED_PALETTES.get(column, {})
    missing_categories = [category for category in categories if category not in fixed]
    generated_colors = sns.color_palette('tab20', n_colors=max(len(missing_categories), 1))
    generated = dict(zip(missing_categories, generated_colors))
    return {category: fixed[category] if category in fixed else generated[category] for category in categories}

def style_umap_axes(ax, dataframe, title):
    x_range = dataframe['UMAP1'].max() - dataframe['UMAP1'].min()
    y_range = dataframe['UMAP2'].max() - dataframe['UMAP2'].min()
    ax.set_xlim(dataframe['UMAP1'].min() - 0.08 * x_range, dataframe['UMAP1'].max() + 0.08 * x_range)
    ax.set_ylim(dataframe['UMAP2'].min() - 0.08 * y_range, dataframe['UMAP2'].max() + 0.08 * y_range)
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.set_title(title, fontsize=10, pad=8)
    ax.set_xticks([])
    ax.set_yticks([])
    sns.despine(ax=ax, top=True, right=True, left=True, bottom=True)
UMAP_FIGSIZE = (5.5, 5)
UMAP_AX_POSITION = [0.18, 0.2, 0.62, 0.62]

def create_fixed_umap_figure():
    fig = plt.figure(figsize=UMAP_FIGSIZE, facecolor='white')
    ax = fig.add_axes(UMAP_AX_POSITION)
    return (fig, ax)

def plot_metadata_on_umap(dataframe, metadata_column, point_size=7):
    if metadata_column not in dataframe.columns:
        raise ValueError(f'{metadata_column!r} not found in the UMAP table')
    plot_df = dataframe.copy()
    plot_df['_display_category'] = metadata_display_values(plot_df, metadata_column)
    categories = ordered_categories(plot_df['_display_category'])
    palette = category_palette(metadata_column, categories)
    (fig, ax) = create_fixed_umap_figure()
    for category in categories:
        category_data = plot_df.loc[plot_df['_display_category'].eq(category)]
        is_wild_type = category == 'Wild type'
        ax.scatter(category_data['UMAP1'], category_data['UMAP2'], label=category, color=palette[category], s=point_size + 3 if is_wild_type else point_size, alpha=0.95 if is_wild_type else 0.82, linewidths=0.35 if is_wild_type else 0, edgecolors='white' if is_wild_type else 'none', rasterized=False, zorder=20 if is_wild_type else 5)
    if wt_density is not None:
        ax.contour(wt_density['x_mesh'], wt_density['y_mesh'], wt_density['values'], levels=[wt_density['thresholds'][WT_CONTOUR_PROBABILITY]], colors='#222222', linewidths=0.65, alpha=0.65, zorder=15)
    display_name = METADATA_LABELS.get(metadata_column, metadata_column)
    style_umap_axes(ax, plot_df, display_name)
    legend_columns = 2 if len(categories) > 10 else 1
    ax.legend(title=display_name, bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, fontsize=7, title_fontsize=8, markerscale=1.4, ncol=legend_columns, borderaxespad=0)
    plt.show()
    plt.close(fig)
    return (fig, ax)

In [ ]:
for metadata_column in PLOT_METADATA_COLUMNS:
    file_stem = metadata_column.removeprefix('Metadata_')
    plot_metadata_on_umap(umap_df, metadata_column=metadata_column)

## Gene classification and composition
Genes represented on multiple plates use the mean coordinate before density classification.

In [ ]:
outer_contour_threshold = wt_density['thresholds'][WT_CONTOUR_PROBABILITY]
mutant_umap = umap_df.loc[umap_df['Metadata_gene'].ne(WT_LABEL)].groupby('Metadata_gene', as_index=False).agg(UMAP1=('UMAP1', 'mean'), UMAP2=('UMAP2', 'mean')).merge(gene_metadata, on='Metadata_gene', validate='one_to_one')
mutant_umap['WT_density'] = wt_density['kde'](mutant_umap[['UMAP1', 'UMAP2']].to_numpy().T)
mutant_umap['outside_outer_WT_contour'] = mutant_umap['WT_density'].lt(outer_contour_threshold)
total_mutants = len(mutant_umap)
if not total_mutants or total_mutants != len(profile_genes):
    raise ValueError('Gene counts do not match the cleaned profile data')
total_outside = int(mutant_umap['outside_outer_WT_contour'].sum())
total_inside = total_mutants - total_outside
all_mask = pd.Series(True, index=mutant_umap.index)

In [ ]:
AXIS_COLORS = {'RpoS': '#E69F00', 'ppGpp': '#0072B2', 'RpoS and ppGpp': '#009E73', 'No documented association': '#BDBDBD'}
composition_rows = []
(fig, axes) = plt.subplots(1, 3, figsize=(13.8, 5.4), facecolor='white')
for (ax, (label, selection)) in zip(axes, [('All deletion genes', all_mask), ('Outside WT contour', mutant_umap['outside_outer_WT_contour']), ('Inside WT contour', ~mutant_umap['outside_outer_WT_contour'])]):
    group = mutant_umap.loc[selection]
    counts = group['Metadata_stationary_phase_axis'].value_counts().reindex(AXIS_ORDER, fill_value=0)
    assert counts.sum() == len(group)
    for (category, count) in counts.items():
        composition_rows.append({'Phenotype_group': label, 'Regulatory_axis': category, 'n_genes': int(count), 'Denominator': len(group), 'Percent': 100 * count / len(group) if len(group) else np.nan})
    ax.set_title(label, fontsize=12, pad=15)
    nonzero = counts.loc[counts.gt(0)]
    if nonzero.empty:
        ax.text(0.5, 0.5, 'No genes', transform=ax.transAxes, ha='center', va='center')
        ax.axis('off')
        continue
    (wedges, _) = ax.pie(nonzero, colors=[AXIS_COLORS[c] for c in nonzero.index], startangle=90, counterclock=False, wedgeprops={'width': 0.43, 'edgecolor': 'white', 'linewidth': 1.2})
    ax.text(0, 0, f'n = {len(group):,}\ngenes', ha='center', va='center', fontsize=11)
    ax.legend(wedges, [f"{(cat + ' only' if cat in ['RpoS', 'ppGpp'] else cat)}: {count:,} ({100 * count / len(group):.1f}%)" for (cat, count) in nonzero.items()], loc='upper center', bbox_to_anchor=(0.5, -0.03), frameon=False, fontsize=9)
fig.suptitle('Regulator annotation composition of morphological phenotype groups', fontsize=13, y=0.97)
fig.subplots_adjust(top=0.82, bottom=0.25, wspace=0.35)
plt.show()
plt.close(fig)
regulatory_composition = pd.DataFrame(composition_rows)
display(regulatory_composition.round(2))

In [ ]:
if not RECOMPUTE_UMAP:
    np.testing.assert_array_equal(regulatory_composition.n_genes, reference_pies.n_genes)